In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Legal Clause Analyzer — Retrieve & Assess Contract Clauses with RAG

## 1. Project Overview

**Task:** Index contract clauses, retrieve relevant ones for legal questions, and flag risky or missing clause patterns — all using local models.

**Approach:** Build sample contracts → parse into clauses → embed with sentence-transformers → store in ChromaDB → retrieve for questions → LLM-based risk assessment with pattern matching.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for analysis
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store
- `LangChain` — text splitting, orchestration

> **⚠️ Important Disclaimer:** This notebook is for **educational purposes only**. It is NOT a substitute for professional legal advice. LLMs can hallucinate, misinterpret legal language, and miss critical nuances. Always consult a qualified attorney for real contract review. See Section 27 for a full limitations discussion.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Parse contracts** into structured clauses with metadata |
| 2 | **Embed legal text** and build a searchable clause index |
| 3 | **Retrieve clauses** relevant to specific legal questions |
| 4 | **Assess risk** — flag vague, one-sided, or aggressive clauses |
| 5 | **Detect missing clauses** — compare against a standard checklist |
| 6 | **Understand limitations** — know when NOT to trust LLM outputs |

## 3. Problem Statement

Contracts are long, dense, and full of legal jargon. Even experienced professionals miss problematic clauses. Common pain points:

- **Finding relevant clauses:** "Where does this contract talk about liability?"
- **Spotting risk:** "Is this indemnification clause unusually broad?"
- **Checking completeness:** "Are standard protection clauses present?"

This notebook builds a RAG system that retrieves relevant clauses and uses an LLM to flag potential risks — while being transparent about what it CAN'T do.

## 4. Why This Project Matters

- **Practical RAG over structured documents** — contracts have clear section boundaries
- **Risk assessment prompting** — learn pattern-based analysis with LLMs
- **Responsible AI** — explicitly model limitations and confidence
- **Local & private** — contract data never leaves your machine

## 5. Pipeline Architecture

```
Sample Contracts (structured text with clause headings)
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 1: PARSING                        │
  │  • Split on clause headings              │
  │  • Extract clause type, number, contract │
  │  • Classify clause category              │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 2: EMBEDDING + INDEXING           │
  │  • all-MiniLM-L6-v2 (384-dim)          │
  │  • ChromaDB with metadata filters       │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 3: RETRIEVAL                      │
  │  • Semantic search for relevant clauses  │
  │  • Filter by contract / clause type      │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 4: RISK ASSESSMENT                │
  │  • LLM flags vague, one-sided, broad    │
  │  • Pattern matching for red flags        │
  │  • Confidence + limitations output       │
  └──────────────────────────────────────────┘
         │
  ┌──────┴──────────────────────────────────┐
  │  Stage 5: MISSING CLAUSE DETECTION       │
  │  • Compare against standard checklist    │
  │  • Report gaps with explanations         │
  └──────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface langchain-text-splitters chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings
from pathlib import Path
from typing import Optional
from collections import Counter

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

print("All imports OK")

## 8. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 5
TEMP_ANALYSIS = 0.1      # Low temp for faithful legal analysis
TEMP_JUDGE = 0.0          # Zero temp for structured evaluation

print("Configuration")
print(f"  LLM: {LLM_MODEL}")
print(f"  Embeddings: {EMBED_MODEL}")
print(f"  Chunk: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Retrieval: top-{TOP_K}")
print(f"  Temperature: {TEMP_ANALYSIS} (analysis), {TEMP_JUDGE} (judge)")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


def wrap_print(text: str, width: int = 90):
    """Print text with word wrapping."""
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Contract Data & Parsing

## 10. Sample Contracts

We define three realistic (but fictional) contracts. In production, you'd load real PDFs or DOCX files.

> **Note:** These are simplified for teaching. Real contracts are much longer and more complex.

In [ ]:
SAMPLE_CONTRACTS = {
    "saas_agreement.md": """# SaaS Service Agreement
## Contract ID: SA-2026-001
## Parties: Acme Corp (Provider) and Beta Inc (Customer)
## Effective Date: January 1, 2026

### 1. Service Description
Provider shall deliver a cloud-based project management platform (\"the Service\")
to Customer. The Service includes task tracking, team collaboration, file storage
(up to 100GB), and API access. Provider reserves the right to modify features
with 30 days written notice.

### 2. Service Level Agreement (SLA)
Provider guarantees 99.5% monthly uptime, measured as the percentage of minutes
the Service is available during a calendar month. Scheduled maintenance windows
(Sundays 2:00-6:00 AM UTC) are excluded from uptime calculations. If uptime falls
below 99.5%, Customer is entitled to service credits equal to 5% of monthly fees
per percentage point below the target, capped at 30% of monthly fees.

### 3. Payment Terms
Customer shall pay $5,000 per month, due within 30 days of invoice date. Late
payments incur interest at 1.5% per month. Provider may suspend service after 60
days of non-payment with 15 days written notice. All fees are non-refundable
except as expressly stated in Section 2 (SLA credits).

### 4. Data Handling and Privacy
Provider shall process Customer data only as necessary to deliver the Service.
All data is encrypted at rest (AES-256) and in transit (TLS 1.2+). Provider will
not access Customer data except for troubleshooting with prior written consent.
Upon termination, Provider shall delete Customer data within 30 days unless legally
required to retain it.

### 5. Intellectual Property
All pre-existing IP remains the property of the respective party. Customer retains
full ownership of all data uploaded to the Service. Provider retains ownership of
the Service platform, including all improvements and modifications.

### 6. Limitation of Liability
PROVIDER'S TOTAL LIABILITY SHALL NOT EXCEED THE FEES PAID BY CUSTOMER IN THE
TWELVE (12) MONTHS PRECEDING THE CLAIM. IN NO EVENT SHALL PROVIDER BE LIABLE FOR
INDIRECT, INCIDENTAL, CONSEQUENTIAL, SPECIAL, OR EXEMPLARY DAMAGES, INCLUDING
LOST PROFITS, LOST DATA, OR BUSINESS INTERRUPTION, REGARDLESS OF THE THEORY OF
LIABILITY. This limitation applies even if Provider has been advised of the
possibility of such damages.

### 7. Indemnification
Provider shall indemnify Customer against third-party claims alleging that the
Service infringes any patent or copyright. Customer shall indemnify Provider against
claims arising from Customer's use of the Service in violation of applicable law or
this Agreement. The indemnifying party shall have sole control of the defense.

### 8. Termination
Either party may terminate with 90 days written notice. Provider may terminate
immediately if Customer breaches any material term and fails to cure within 30 days
of written notice. Upon termination, the provisions of Sections 5, 6, 7, and 9
shall survive.

### 9. Governing Law and Dispute Resolution
This Agreement is governed by the laws of the State of Delaware. Any disputes shall
be resolved through binding arbitration administered by the American Arbitration
Association in Wilmington, Delaware. Each party bears its own attorneys' fees.

### 10. Confidentiality
Each party agrees to hold the other's Confidential Information in strict confidence
for a period of 3 years after disclosure. Confidential Information excludes
information that is publicly available, independently developed, or rightfully
received from a third party without restriction.

### 11. Force Majeure
Neither party shall be liable for delays or failures caused by events beyond
reasonable control, including natural disasters, war, government actions, or
internet service failures. The affected party must notify the other within 5
business days and make reasonable efforts to mitigate the impact.""",


    "employment_contract.md": """# Employment Agreement
## Contract ID: EA-2026-042
## Parties: TechStart LLC (Employer) and Jane Smith (Employee)
## Effective Date: March 1, 2026

### 1. Position and Duties
Employee is hired as Senior Software Engineer, reporting to the VP of Engineering.
Employee shall devote full working time and best efforts to Employer's business.
Employee shall perform all duties as reasonably assigned by Employer, which may
be modified at Employer's sole discretion.

### 2. Compensation
Base salary: $185,000 per year, paid semi-monthly. Performance bonus: up to 20%
of base salary, determined at Employer's sole discretion based on individual and
company performance. Salary reviews occur annually but Employer is not obligated
to increase compensation.

### 3. Benefits
Employee is eligible for health insurance, dental, vision, and 401(k) with 4%
employer match after 90 days. 15 days paid time off (PTO) per year, accruing
monthly. Unused PTO does not carry over to the next calendar year and is not
paid out upon termination.

### 4. Intellectual Property Assignment
ALL work product, inventions, discoveries, and improvements conceived or developed
by Employee during employment — whether or not during working hours, whether or
not using Employer's resources — shall be the sole and exclusive property of
Employer. Employee hereby irrevocably assigns all rights, title, and interest in
such work product to Employer. This includes any work related to Employer's
current or reasonably anticipated future business.

### 5. Non-Compete
Employee agrees that for a period of 24 months after termination (for any reason),
Employee shall not directly or indirectly engage in, own, manage, or consult for
any business that competes with Employer's business within North America. Employee
acknowledges this restriction is reasonable and necessary to protect Employer's
legitimate business interests.

### 6. Non-Solicitation
For a period of 18 months after termination, Employee shall not directly or
indirectly solicit, recruit, or hire any employee or contractor of Employer, or
solicit any customer or client of Employer for a competing business.

### 7. Confidentiality
Employee agrees to hold all proprietary information in strict confidence during
and after employment, with no time limitation. Proprietary information includes
business plans, customer lists, financial data, technical designs, source code,
algorithms, and any information designated as confidential by Employer. Employee
shall return all materials upon termination.

### 8. Termination
Employer may terminate Employee at any time, with or without cause, with 2 weeks
notice (or pay in lieu of notice). Employee may resign with 4 weeks notice. Upon
termination for cause, all unvested benefits are forfeited. Sections 4, 5, 6,
and 7 survive termination.

### 9. Dispute Resolution
Any disputes shall be resolved through binding arbitration under the rules of
JAMS in San Francisco, California. Employee waives the right to a jury trial and
the right to participate in class action proceedings. Each party bears its own
costs. The arbitrator's decision is final and non-appealable.

### 10. Governing Law
This Agreement is governed by California law, without regard to conflict of laws
principles. If any provision is found unenforceable, the remaining provisions
shall continue in full force.""",


    "vendor_nda.md": """# Mutual Non-Disclosure Agreement
## Contract ID: NDA-2026-117
## Parties: GlobalTech Inc (Disclosing Party) and DataFlow Ltd (Receiving Party)
## Effective Date: February 15, 2026

### 1. Definition of Confidential Information
\"Confidential Information\" means any and all information disclosed by either party,
whether orally, in writing, or electronically, that is designated as confidential
or that a reasonable person would understand to be confidential. This includes
technical data, trade secrets, business plans, customer information, financial
records, product roadmaps, and source code.

### 2. Obligations of Receiving Party
Receiving Party shall: (a) hold Confidential Information in strict confidence;
(b) not disclose to any third party without prior written consent; (c) use
Confidential Information solely for the Purpose (evaluating a potential business
partnership); (d) restrict access to employees with a need to know who are bound
by confidentiality obligations no less protective than this Agreement.

### 3. Exclusions
Confidential Information does not include information that: (a) is or becomes
publicly available without breach; (b) was known to Receiving Party prior to
disclosure; (c) is independently developed without reference to Confidential
Information; (d) is received from a third party without restriction.

### 4. Term and Duration
This Agreement is effective for 2 years from the Effective Date. The
confidentiality obligations shall survive for 5 years after the last disclosure
of Confidential Information, regardless of termination.

### 5. Return of Materials
Upon termination or request, Receiving Party shall promptly return or destroy all
Confidential Information and certify destruction in writing. Receiving Party may
retain one archival copy solely for legal compliance purposes.

### 6. Remedies
Both parties acknowledge that breach may cause irreparable harm for which monetary
damages are inadequate. The Disclosing Party shall be entitled to seek injunctive
relief in addition to any other remedies available at law or in equity.

### 7. No License or Warranty
Nothing in this Agreement grants any license under any patent, copyright, or other
intellectual property right. ALL CONFIDENTIAL INFORMATION IS PROVIDED \"AS IS\"
WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED.

### 8. Governing Law
This Agreement is governed by the laws of the State of New York. Any disputes shall
be resolved in the state or federal courts located in New York County.""",
}

print(f"Sample contracts: {len(SAMPLE_CONTRACTS)}")
for name, content in SAMPLE_CONTRACTS.items():
    num_sections = len(re.findall(r"^### \d+", content, re.MULTILINE))
    print(f"  📄 {name}: {len(content):,} chars, {num_sections} clauses")

## 11. Contract Parsing — Extracting Clauses

Each clause becomes a separate chunk with rich metadata: contract name, clause number, clause title, and auto-detected category.

In [ ]:
# Clause category lookup — maps common heading keywords to categories
CLAUSE_CATEGORIES = {
    "service": "obligations", "description": "obligations", "duties": "obligations",
    "position": "obligations", "sla": "performance",
    "payment": "financial", "compensation": "financial", "fees": "financial",
    "benefit": "financial",
    "data": "privacy", "privacy": "privacy", "confidential": "confidentiality",
    "intellectual": "ip", "ip": "ip", "license": "ip",
    "liability": "liability", "limitation": "liability",
    "indemnif": "indemnification",
    "terminat": "termination",
    "governing": "jurisdiction", "dispute": "jurisdiction", "arbitrat": "jurisdiction",
    "non-compete": "restrictive_covenant", "non-solicitation": "restrictive_covenant",
    "force majeure": "force_majeure",
    "exclusion": "definitions", "definition": "definitions",
    "return": "termination", "remedy": "remedies", "remedies": "remedies",
    "warrant": "warranty", "no license": "ip",
}


def classify_clause(heading: str) -> str:
    """Auto-detect clause category from its heading."""
    heading_lower = heading.lower()
    for keyword, category in CLAUSE_CATEGORIES.items():
        if keyword in heading_lower:
            return category
    return "general"


def parse_contract(filename: str, content: str) -> list[dict]:
    """Parse a contract into individual clauses with metadata."""
    # Extract contract-level metadata from header
    contract_name = filename.replace(".md", "").replace("_", " ").title()
    parties_match = re.search(r"Parties:\s*(.+)", content)
    parties = parties_match.group(1).strip() if parties_match else "Unknown"
    date_match = re.search(r"Effective Date:\s*(.+)", content)
    eff_date = date_match.group(1).strip() if date_match else "Unknown"

    # Split on ### heading pattern (clause level)
    parts = re.split(r"(?=^### \d+)", content, flags=re.MULTILINE)

    clauses = []
    for part in parts:
        part = part.strip()
        if not part or not re.match(r"^### \d+", part):
            continue

        heading_match = re.match(r"^### (\d+)\.\s*(.+)", part)
        if not heading_match:
            continue

        clause_num = heading_match.group(1)
        clause_title = heading_match.group(2).strip()
        # Remove the heading from the body text
        body = re.sub(r"^### .+\n", "", part).strip()
        category = classify_clause(clause_title)

        clauses.append({
            "text": part,  # Keep full text including heading
            "body": body,
            "metadata": {
                "contract": contract_name,
                "filename": filename,
                "parties": parties,
                "effective_date": eff_date,
                "clause_num": clause_num,
                "clause_title": clause_title,
                "category": category,
            },
        })

    return clauses


# Parse all contracts
all_clauses = []
for fname, content in SAMPLE_CONTRACTS.items():
    parsed = parse_contract(fname, content)
    all_clauses.extend(parsed)

print(f"Parsed {len(all_clauses)} clauses from {len(SAMPLE_CONTRACTS)} contracts\n")

# Show summary
for c in all_clauses:
    m = c["metadata"]
    print(f"  §{m['clause_num']:>2} [{m['category']:>20}] "
          f"{m['contract'][:25]:25s} → {m['clause_title']}")

## 12. Chunking Clauses

Short clauses stay as-is. Long clauses get chunked with overlap to fit the embedding model.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)

all_chunks = []

for clause in all_clauses:
    text = clause["text"]
    metadata = clause["metadata"]

    if len(text) <= CHUNK_SIZE:
        all_chunks.append({"text": text, "metadata": {**metadata, "chunk_index": 0, "chunk_total": 1}})
    else:
        chunks = splitter.split_text(text)
        for i, ch in enumerate(chunks):
            all_chunks.append({"text": ch, "metadata": {**metadata, "chunk_index": i, "chunk_total": len(chunks)}})

print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunk length: {sum(len(c['text']) for c in all_chunks) / len(all_chunks):.0f} chars")

# Breakdown by contract
contract_counts = Counter(c["metadata"]["contract"] for c in all_chunks)
print(f"\nChunks per contract:")
for name, count in contract_counts.items():
    print(f"  {name}: {count}")

# Breakdown by category
cat_counts = Counter(c["metadata"]["category"] for c in all_chunks)
print(f"\nChunks per category:")
for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}")

---

# Part B — Embedding, Indexing & Retrieval

## 13. Build the Vector Store

In [ ]:
# Initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# Initialize ChromaDB
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("legal_clauses")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="legal_clauses",
    metadata={"hnsw:space": "cosine"},
)

# Embed and index
texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

print("Embedding all clause chunks...")
all_embeds = []
batch_size = 32
for i in range(0, len(texts), batch_size):
    batch = texts[i : i + batch_size]
    all_embeds.extend(embeddings.embed_documents(batch))
    print(f"  Batch {i // batch_size + 1}/{(len(texts) - 1) // batch_size + 1} done")

collection.add(
    documents=texts, embeddings=all_embeds,
    metadatas=metadatas, ids=ids,
)
print(f"\n✅ Vector store built: {collection.count()} chunks indexed")

## 14. Retrieval Functions

In [ ]:
def retrieve(query: str, top_k: int = TOP_K,
             where: Optional[dict] = None) -> list[dict]:
    """Retrieve top-k clause chunks for a query."""
    query_embed = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_embed], "n_results": top_k}
    if where:
        kwargs["where"] = where
    results = collection.query(**kwargs)

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return chunks


def display_chunks(chunks: list[dict], max_text: int = 120):
    """Pretty-print retrieved clause chunks."""
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        sim = 1 - c["distance"]
        print(f"\n  [{i}] (sim={sim:.3f}) {m['contract']} §{m['clause_num']}: "
              f"{m['clause_title']}")
        print(f"      Category: {m['category']} | Parties: {m['parties'][:40]}")
        print(f"      {c['text'][:max_text]}...")


# Test retrieval
print("BASIC RETRIEVAL TEST")
print("=" * 60)

q = "What happens if the provider fails to meet uptime guarantees?"
print(f"Q: {q}")
chunks = retrieve(q, top_k=3)
display_chunks(chunks)

## 15. Filtered Retrieval — By Contract and Category

In [ ]:
print("FILTERED RETRIEVAL EXAMPLES")
print("=" * 60)

# Filter by category
q = "What are the liability limitations?"
print(f"\nQ: {q}")
print("Filter: category = 'liability'")
chunks = retrieve(q, where={"category": "liability"})
display_chunks(chunks[:3])

# Filter by contract
q = "What happens at termination?"
print(f"\n{'─'*60}")
print(f"Q: {q}")
print("Filter: contract = 'Employment Contract'")
chunks = retrieve(q, where={"contract": "Employment Contract"})
display_chunks(chunks[:3])

# Compound filter: specific contract + category
q = "What IP rights are assigned?"
print(f"\n{'─'*60}")
print(f"Q: {q}")
print("Filter: category = 'ip'")
chunks = retrieve(q, where={"category": "ip"})
display_chunks(chunks[:3])

---

# Part C — Risk Assessment

## 16. Clause Risk Analysis

We prompt the LLM to analyze each clause for common risk patterns:

| Pattern | Description | Example |
|---------|-------------|--------|
| **Vague language** | Terms like "reasonable", "may", "at discretion" without definition | "at Employer's sole discretion" |
| **One-sided terms** | Benefits only one party disproportionately | Unlimited IP assignment |
| **Overly broad scope** | Geographic, temporal, or subject scope too wide | "24 months, North America" non-compete |
| **Missing protections** | Expected safeguard absent | No data breach notification |
| **Aggressive penalties** | Disproportionate consequences | "all unvested benefits forfeited" |

> **⚠️ Limitation:** The LLM is pattern-matching against common legal templates. It cannot understand jurisdiction-specific enforceability, case law, or regulatory requirements. Its risk flags are *suggestions for human review*, not legal conclusions.

In [ ]:
RISK_SYSTEM = """You are a legal clause reviewer for educational purposes.
Analyze contract clauses for potential risks and concerns.

CRITICAL LIMITATIONS — you MUST follow these:
1. You are NOT a lawyer. Your analysis is educational only.
2. Flag patterns, but never make legal conclusions about enforceability.
3. Always recommend professional legal review for flagged items.
4. Do not guess jurisdiction-specific rules — say you don't know.
5. Be balanced — note both risks AND protective elements."""

RISK_PROMPT = """Analyze this contract clause for potential risk patterns.

CONTRACT: {contract}
CLAUSE §{clause_num}: {clause_title}
CATEGORY: {category}

CLAUSE TEXT:
{text}

Analyze for these patterns:
- VAGUE LANGUAGE: undefined terms like "reasonable", "sole discretion", "may"
- ONE-SIDED TERMS: benefits one party disproportionately
- OVERLY BROAD SCOPE: time, geography, or subject scope that seems excessive
- MISSING PROTECTIONS: safeguards you'd normally expect
- AGGRESSIVE PENALTIES: disproportionate consequences

Return ONLY JSON:
{{"risk_level": "high|medium|low",
  "flags": [
    {{"pattern": "pattern name",
      "detail": "specific concern",
      "quote": "exact text from clause",
      "suggestion": "what to look for or negotiate"}}
  ],
  "protective_elements": ["list of good protections already present"],
  "overall_note": "one-sentence summary",
  "needs_lawyer": true}}"""


def assess_clause_risk(clause: dict) -> dict:
    """Analyze a single clause for risk patterns."""
    m = clause["metadata"]
    resp = ask(
        RISK_PROMPT.format(
            contract=m["contract"], clause_num=m["clause_num"],
            clause_title=m["clause_title"], category=m["category"],
            text=clause["body"] if "body" in clause else clause["text"],
        ),
        system=RISK_SYSTEM,
        temperature=TEMP_ANALYSIS,
    )
    result = parse_json(resp)
    if result:
        result["clause_title"] = m["clause_title"]
        result["contract"] = m["contract"]
        result["clause_num"] = m["clause_num"]
    else:
        result = {"raw": resp, "clause_title": m["clause_title"],
                  "risk_level": "unknown"}
    return result


print("assess_clause_risk() defined")

## 17. Analyze High-Interest Clauses

In [ ]:
# Select clauses commonly flagged in contract reviews
interest_categories = ["liability", "ip", "restrictive_covenant",
                       "indemnification", "termination"]

target_clauses = [c for c in all_clauses if c["metadata"]["category"] in interest_categories]

print(f"RISK ASSESSMENT — {len(target_clauses)} clauses")
print("=" * 70)

risk_results = []

for clause in target_clauses:
    m = clause["metadata"]
    print(f"\n  Analyzing: {m['contract']} §{m['clause_num']} — {m['clause_title']}...", end=" ")
    result = assess_clause_risk(clause)
    risk_results.append(result)

    level = result.get("risk_level", "?")
    badge = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(level, "⚪")
    print(f"{badge} {level.upper()}")

# Summary table
print(f"\n{'='*70}")
print(f"RISK SUMMARY")
print(f"{'='*70}")

for r in sorted(risk_results, key=lambda x: {"high":0,"medium":1,"low":2}.get(x.get("risk_level",""),3)):
    level = r.get("risk_level", "?")
    badge = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(level, "⚪")
    flags = r.get("flags", [])
    print(f"\n  {badge} {r.get('contract','')}: §{r.get('clause_num','')} {r.get('clause_title','')}")
    print(f"     {r.get('overall_note', '')}")
    for f in flags[:2]:
        print(f"     ⚠ {f.get('pattern','')}: {f.get('detail','')[:70]}")

## 18. Deep Dive — One Clause

In [ ]:
# Pick the non-compete clause for a detailed review
nc_clause = next(c for c in all_clauses
                 if c["metadata"]["category"] == "restrictive_covenant"
                 and "non-compete" in c["metadata"]["clause_title"].lower())

result = assess_clause_risk(nc_clause)

print("DEEP DIVE: NON-COMPETE CLAUSE")
print("=" * 70)
m = nc_clause["metadata"]
print(f"Contract: {m['contract']}")
print(f"Clause: §{m['clause_num']} — {m['clause_title']}")
print(f"\nFull text:")
wrap_print(nc_clause["body"])

level = result.get("risk_level", "?")
badge = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(level, "⚪")
print(f"\n{'─'*70}")
print(f"Risk Level: {badge} {level.upper()}")
print(f"Note: {result.get('overall_note', '')}")

print(f"\nFlags:")
for f in result.get("flags", []):
    print(f"  ⚠ {f.get('pattern', '')}:")
    print(f"    Detail: {f.get('detail', '')}")
    if f.get("quote"):
        print(f"    Quote: \"{f['quote'][:80]}\"")
    if f.get("suggestion"):
        print(f"    → {f['suggestion']}")

print(f"\nProtective elements:")
for p in result.get("protective_elements", []):
    print(f"  ✅ {p}")

print(f"\n⚠️ Needs lawyer review: {result.get('needs_lawyer', True)}")

---

# Part D — Missing Clause Detection

## 19. Standard Clause Checklist

Different contract types have expected clauses. If a key clause is missing, that's a risk worth flagging.

In [ ]:
# Standard clauses expected by contract type
STANDARD_CHECKLISTS = {
    "Saas Agreement": {
        "obligations": "Service description or scope of services",
        "performance": "Service Level Agreement (SLA) with uptime guarantees",
        "financial": "Payment terms and fee schedule",
        "privacy": "Data handling, privacy, and security commitments",
        "ip": "Intellectual property ownership",
        "liability": "Limitation of liability with cap",
        "indemnification": "Mutual indemnification terms",
        "termination": "Termination conditions and notice period",
        "jurisdiction": "Governing law and dispute resolution",
        "confidentiality": "Confidentiality obligations",
        "force_majeure": "Force majeure / acts of God",
        "data_breach": "Data breach notification requirements",
        "insurance": "Insurance requirements",
        "audit": "Audit rights for the customer",
    },
    "Employment Contract": {
        "obligations": "Position, duties, and reporting structure",
        "financial": "Compensation and benefits",
        "ip": "IP assignment or work-for-hire terms",
        "restrictive_covenant": "Non-compete / non-solicitation",
        "confidentiality": "Confidentiality / NDA obligations",
        "termination": "Termination terms and notice period",
        "jurisdiction": "Governing law and dispute resolution",
        "severance": "Severance package terms",
        "equity": "Stock options or equity vesting",
        "whistleblower": "Whistleblower / retaliation protections",
        "disability": "Disability and leave accommodations",
    },
    "Vendor Nda": {
        "definitions": "Definition of Confidential Information",
        "confidentiality": "Obligations of receiving party",
        "definitions_excl": "Exclusions from confidential info",
        "termination": "Term and survival period",
        "remedies": "Remedies for breach (injunctive relief)",
        "ip": "No license granted / IP disclaimer",
        "jurisdiction": "Governing law",
        "return_materials": "Return or destruction of materials",
        "permitted_disclosures": "Permitted disclosures (legal, regulatory)",
    },
}


def detect_missing_clauses(contract_name: str, clauses: list[dict]) -> dict:
    """Compare contract clauses against the standard checklist."""
    # Normalize contract name for lookup
    lookup_name = contract_name
    checklist = STANDARD_CHECKLISTS.get(lookup_name, {})
    if not checklist:
        for key in STANDARD_CHECKLISTS:
            if key.lower() in contract_name.lower():
                checklist = STANDARD_CHECKLISTS[key]
                lookup_name = key
                break

    if not checklist:
        return {"contract": contract_name, "error": "No checklist found"}

    # Get categories present in the contract
    present_categories = set(c["metadata"]["category"] for c in clauses)

    found = {}
    missing = {}

    for cat, description in checklist.items():
        if cat in present_categories:
            matching = [c for c in clauses if c["metadata"]["category"] == cat]
            found[cat] = {
                "description": description,
                "clause": matching[0]["metadata"]["clause_title"],
            }
        else:
            missing[cat] = {"description": description}

    return {
        "contract": contract_name,
        "total_expected": len(checklist),
        "found": len(found),
        "missing": len(missing),
        "found_clauses": found,
        "missing_clauses": missing,
        "coverage_pct": len(found) / len(checklist) * 100 if checklist else 0,
    }


print("detect_missing_clauses() defined")

## 20. Run Missing Clause Detection

In [ ]:
# Group clauses by contract
contracts_grouped = {}
for c in all_clauses:
    name = c["metadata"]["contract"]
    contracts_grouped.setdefault(name, []).append(c)

print("MISSING CLAUSE DETECTION")
print("=" * 70)

gap_results = {}

for contract_name, clauses in contracts_grouped.items():
    result = detect_missing_clauses(contract_name, clauses)
    gap_results[contract_name] = result

    coverage = result["coverage_pct"]
    badge = "🟢" if coverage >= 80 else ("🟡" if coverage >= 60 else "🔴")

    print(f"\n{badge} {contract_name}")
    print(f"   Coverage: {result['found']}/{result['total_expected']} "
          f"({coverage:.0f}%)")

    if result["missing_clauses"]:
        print(f"   ❌ Missing:")
        for cat, info in result["missing_clauses"].items():
            print(f"      • {cat}: {info['description']}")
    else:
        print(f"   ✅ All expected clauses present")

    print(f"   ✅ Found:")
    for cat, info in list(result["found_clauses"].items())[:5]:
        print(f"      • {cat} → §{info['clause']}")
    if len(result["found_clauses"]) > 5:
        print(f"      ... and {len(result['found_clauses'])-5} more")

## 21. LLM-Enhanced Gap Analysis

In [ ]:
GAP_PROMPT = """A {contract_type} contract is MISSING the following standard clauses:

{missing_list}

For each missing clause, explain:
1. What this clause normally covers
2. Why its absence could be risky
3. What questions to raise with a lawyer

Return ONLY JSON:
{{"gaps": [
    {{"clause_type": "name",
      "risk_of_absence": "what could go wrong",
      "typical_coverage": "what this clause normally says",
      "question_for_lawyer": "what to ask",
      "risk_level": "high|medium|low"}}
  ],
  "overall_risk": "summary of combined gap risk",
  "disclaimer": "Always needed: this is..."}}"""

# Analyze the SaaS agreement's gaps
saas_result = gap_results.get("Saas Agreement", {})
if saas_result.get("missing_clauses"):
    missing_list = "\n".join(
        f"- {cat}: {info['description']}"
        for cat, info in saas_result["missing_clauses"].items()
    )

    resp = ask(
        GAP_PROMPT.format(contract_type="SaaS Service", missing_list=missing_list),
        system=RISK_SYSTEM,
        temperature=TEMP_ANALYSIS,
    )
    gap_analysis = parse_json(resp)

    print("LLM-ENHANCED GAP ANALYSIS — SaaS Agreement")
    print("=" * 70)

    if gap_analysis:
        for g in gap_analysis.get("gaps", []):
            level = g.get("risk_level", "?")
            badge = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(level, "⚪")
            print(f"\n  {badge} {g.get('clause_type', '')}")
            print(f"     Normally covers: {g.get('typical_coverage', '')[:80]}")
            print(f"     Risk if absent: {g.get('risk_of_absence', '')[:80]}")
            print(f"     Ask lawyer: {g.get('question_for_lawyer', '')[:80]}")

        print(f"\n{'─'*70}")
        print(f"Overall: {gap_analysis.get('overall_risk', '')}")
        print(f"⚠️ {gap_analysis.get('disclaimer', 'This is educational only.')}")
    else:
        wrap_print(resp[:400])
else:
    print("No missing clauses in the SaaS agreement — skipping gap analysis.")

---

# Part E — RAG Q&A Over Contracts

## 22. Question Answering Pipeline

In [ ]:
QA_SYSTEM = """You are a contract clause retrieval assistant for educational purposes.
You answer questions using ONLY the contract clauses provided as sources.

CRITICAL RULES:
1. Base answers ONLY on the provided clause text. Never invent terms.
2. Cite sources as [Source: contract §clause_number — clause_title].
3. If the clauses don't answer the question, say so explicitly.
4. Note important caveats or limitations you see.
5. Always end with: \"⚠️ This is for educational purposes. Consult a lawyer for advice.\"
"""

QA_PROMPT = """Answer this question using the contract clauses below.

QUESTION: {question}

CONTRACT CLAUSES:
{sources}

Return ONLY JSON:
{{"answer": "detailed answer with [Source: contract §num] citations",
  "clauses_cited": ["contract §num — title"],
  "confidence": "high|medium|low",
  "caveats": ["important limitations or ambiguities"],
  "disclaimer": "educational purposes statement"}}"""


def format_clause_sources(chunks: list[dict]) -> str:
    parts = []
    for i, c in enumerate(chunks, 1):
        m = c["metadata"]
        parts.append(
            f"[Source {i}: {m['contract']} §{m['clause_num']} — {m['clause_title']}]\n"
            f"{c['text']}\n"
        )
    return "\n".join(parts)


def ask_contracts(question: str, where: Optional[dict] = None) -> dict:
    """Full RAG pipeline for contract questions."""
    chunks = retrieve(question, where=where)
    sources_text = format_clause_sources(chunks)

    resp = ask(
        QA_PROMPT.format(question=question, sources=sources_text),
        system=QA_SYSTEM,
        temperature=TEMP_ANALYSIS,
    )
    result = parse_json(resp)
    if result:
        result["chunks"] = chunks
    else:
        result = {"answer": resp, "confidence": "unknown", "chunks": chunks}
    return result


def display_qa(question: str, result: dict):
    """Pretty-print a Q&A result."""
    conf = result.get("confidence", "?")
    badge = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(conf, "⚪")

    print(f"\n{'═'*70}")
    print(f"  Q: {question}")
    print(f"  Confidence: {badge} {conf}")
    print(f"{'═'*70}")
    wrap_print(result.get("answer", ""))

    caveats = result.get("caveats", [])
    if caveats:
        print(f"\n  Caveats:")
        for c in caveats:
            print(f"    ⚠ {c}")

    print(f"\n  Sources cited:")
    for s in result.get("clauses_cited", []):
        print(f"    📄 {s}")

    print(f"\n  ⚠️ {result.get('disclaimer', 'Educational only — consult a lawyer.')}")


print("ask_contracts() defined")

## 23. Ask Questions About the Contracts

In [ ]:
# Q1: Specific factual question
q1 = "What is the uptime guarantee and what happens if it's not met?"
r1 = ask_contracts(q1)
display_qa(q1, r1)

In [ ]:
# Q2: Cross-contract comparison
q2 = "Compare the termination terms across all three contracts."
r2 = ask_contracts(q2)
display_qa(q2, r2)

In [ ]:
# Q3: Risk-focused question
q3 = "What intellectual property rights does the employee give up?"
r3 = ask_contracts(q3)
display_qa(q3, r3)

In [ ]:
# Q4: Out-of-scope question
q4 = "What is the GDPR compliance status of these contracts?"
r4 = ask_contracts(q4)
display_qa(q4, r4)

conf = r4.get("confidence", "")
print(f"\n  → {'✅' if conf == 'low' else '⚠'} Expected LOW confidence "
      f"(GDPR not explicitly mentioned in these contracts)")

## 24. Batch Contract Review

In [ ]:
batch_questions = [
    "Who bears the attorneys' fees in disputes?",
    "How long do confidentiality obligations last?",
    "What happens to data after the agreement ends?",
    "Can the employer change job duties unilaterally?",
    "What liability cap applies to the SaaS provider?",
    "Is there a non-compete and how long does it last?",
]

print("BATCH CONTRACT Q&A")
print("=" * 60)

for q in batch_questions:
    result = ask_contracts(q)
    conf = result.get("confidence", "?")
    badge = {"high": "🟢", "medium": "🟡", "low": "🔴"}.get(conf, "⚪")
    answer = result.get("answer", "")
    print(f"\n  {badge} Q: {q}")
    print(f"     A: {textwrap.shorten(answer, width=100, placeholder='...')}")

---

# Part F — Evaluation & Experiments

## 25. Retrieval Accuracy

In [ ]:
retrieval_tests = [
    {"query": "What is the uptime SLA?",
     "expected_contract": "Saas Agreement",
     "expected_category": "performance"},
    {"query": "non-compete duration",
     "expected_contract": "Employment Contract",
     "expected_category": "restrictive_covenant"},
    {"query": "What counts as confidential information?",
     "expected_contract": "Vendor Nda",
     "expected_category": "definitions"},
    {"query": "liability cap amount",
     "expected_contract": "Saas Agreement",
     "expected_category": "liability"},
    {"query": "Who owns the inventions?",
     "expected_contract": "Employment Contract",
     "expected_category": "ip"},
    {"query": "Can we get injunctive relief?",
     "expected_contract": "Vendor Nda",
     "expected_category": "remedies"},
]

print("RETRIEVAL ACCURACY")
print("=" * 60)

correct_contract = 0
correct_category = 0

for test in retrieval_tests:
    chunks = retrieve(test["query"], top_k=3)
    top = chunks[0]["metadata"]

    c_match = top["contract"] == test["expected_contract"]
    cat_match = top["category"] == test["expected_category"]
    if c_match:
        correct_contract += 1
    if cat_match:
        correct_category += 1

    status = "✅" if (c_match and cat_match) else ("🟡" if c_match else "❌")
    print(f"\n  {status} Q: {test['query'][:50]}")
    print(f"      Contract: {'✅' if c_match else '❌'} "
          f"expected={test['expected_contract']}, got={top['contract']}")
    print(f"      Category: {'✅' if cat_match else '❌'} "
          f"expected={test['expected_category']}, got={top['category']}")

n = len(retrieval_tests)
print(f"\n{'─'*60}")
print(f"Contract accuracy: {correct_contract}/{n} ({correct_contract/n*100:.0f}%)")
print(f"Category accuracy: {correct_category}/{n} ({correct_category/n*100:.0f}%)")

## 26. Risk Assessment Consistency

In [ ]:
# Run the same clause through the risk analyzer 3 times
# to check if the LLM gives consistent results
consistency_clause = next(c for c in all_clauses
                         if "non-compete" in c["metadata"]["clause_title"].lower())

print("CONSISTENCY CHECK — Non-Compete Clause")
print("=" * 60)
print("Running risk analysis 3 times on the same clause...\n")

runs = []
for trial in range(3):
    r = assess_clause_risk(consistency_clause)
    level = r.get("risk_level", "?")
    num_flags = len(r.get("flags", []))
    flag_patterns = [f.get("pattern", "") for f in r.get("flags", [])]
    runs.append({"level": level, "num_flags": num_flags, "patterns": flag_patterns})
    print(f"  Run {trial+1}: risk={level}, flags={num_flags}, patterns={flag_patterns[:3]}")

# Check consistency
levels = [r["level"] for r in runs]
consistent = len(set(levels)) == 1

print(f"\n{'─'*60}")
print(f"Risk level consistent: {'✅ Yes' if consistent else '⚠️ No'} — {set(levels)}")
print(f"Note: Minor variation in flag details is expected.")
print(f"Major risk-level swings indicate the analysis isn't reliable enough")
print(f"for production use without human review.")

---

## 27. Limitations — What This System CANNOT Do

This is the most important section. Understanding limitations prevents misuse.

| Limitation | Why It Matters | Mitigation |
|-----------|----------------|------------|
| **Not legal advice** | LLMs don't understand law — they pattern-match text | Always consult a qualified attorney |
| **Jurisdiction-blind** | Enforceability varies by state/country (e.g., non-competes are void in CA) | System cannot assess enforceability |
| **No case law** | LLMs don't know how courts have interpreted similar clauses | Only a lawyer can assess precedent |
| **Hallucination risk** | LLM may invent terms or misquote clause text | Always verify citations against originals |
| **Missing context** | Contracts reference external schedules, amendments, side letters | System only sees text you provide |
| **Recency cutoff** | Laws change; LLM training data has a cutoff date | Not a substitute for current legal research |
| **No negotiation skill** | Flagging a risk ≠ knowing how to negotiate a fix | Lawyers understand leverage and trade-offs |
| **Template bias** | Risk patterns are based on common templates; unusual but valid clauses may be flagged | False positives are expected |

### When to Use This System
✅ Initial contract scan to prioritize sections for lawyer review
✅ Learning about common contract structures and clause types
✅ Building a searchable index of your contract library
✅ Identifying which clauses to ask your lawyer about

### When NOT to Use This System
❌ As a substitute for legal review before signing
❌ To assess enforceability of specific terms
❌ To compare against regulatory requirements (GDPR, SOX, etc.)
❌ For contracts in languages other than English
❌ For high-stakes contracts without professional oversight

## 28. Exercises

### Exercise 1: Analyze Your Own Contract

In [ ]:
# ── Exercise 1: Load and analyze a real contract ─────────
# 1. Place your contract text in a variable (redact sensitive info!)
# 2. Parse it with parse_contract()
# 3. Run risk assessment on all clauses
# 4. Check for missing standard clauses

# YOUR_CONTRACT = """
# # Contract Title
# ## Parties: ...
# ### 1. First Clause
# ...
# """
# clauses = parse_contract("my_contract.md", YOUR_CONTRACT)
# for c in clauses:
#     result = assess_clause_risk(c)
#     print(result["risk_level"], result.get("overall_note", ""))

print("Exercise 1: Parse your own contract and run risk assessment.")
print("Remember to redact sensitive information before analysis.")

### Exercise 2: Cross-Contract Risk Comparison

In [ ]:
# ── Exercise 2: Compare the same clause type across contracts ──

# Pick a category and compare clauses across contracts
COMPARE_CATEGORY = "termination"

print(f"CROSS-CONTRACT COMPARISON: {COMPARE_CATEGORY}")
print("=" * 60)

for contract_name, clauses in contracts_grouped.items():
    matching = [c for c in clauses if c["metadata"]["category"] == COMPARE_CATEGORY]
    if not matching:
        print(f"\n  {contract_name}: No {COMPARE_CATEGORY} clause found")
        continue

    clause = matching[0]
    result = assess_clause_risk(clause)
    level = result.get("risk_level", "?")
    badge = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(level, "⚪")

    print(f"\n  {badge} {contract_name} — §{clause['metadata']['clause_num']}")
    print(f"     {result.get('overall_note', '')[:80]}")
    flags = result.get("flags", [])
    if flags:
        print(f"     Top flag: {flags[0].get('pattern','')}: {flags[0].get('detail','')[:60]}")

### Exercise 3: Build a Red-Flag Report

In [ ]:
# ── Exercise 3: Generate a summary red-flag report ──────

REPORT_PROMPT = """Given these risk findings from a contract review, generate a
concise executive summary report.

Contract: {contract}
Findings:
{findings}

Missing clauses:
{missing}

Return ONLY JSON:
{{"executive_summary": "2-3 sentence overview of the contract risk profile",
  "top_risks": ["risk 1", "risk 2", "risk 3"],
  "recommended_actions": ["action 1", "action 2"],
  "overall_risk_rating": "high|medium|low",
  "disclaimer": "Not legal advice. Consult qualified counsel."}}"""

# Generate report for the employment contract
emp_risks = [r for r in risk_results if r.get("contract") == "Employment Contract"]
emp_gaps = gap_results.get("Employment Contract", {})

findings_text = "\n".join(
    f"- §{r.get('clause_num','')}: {r.get('risk_level','?').upper()} — {r.get('overall_note','')}"
    for r in emp_risks
)

missing_text = "\n".join(
    f"- {cat}: {info['description']}"
    for cat, info in emp_gaps.get("missing_clauses", {}).items()
) or "None detected"

resp = ask(
    REPORT_PROMPT.format(
        contract="Employment Contract (TechStart LLC / Jane Smith)",
        findings=findings_text,
        missing=missing_text,
    ),
    system=RISK_SYSTEM,
    temperature=TEMP_ANALYSIS,
)
report = parse_json(resp)

print("RED-FLAG REPORT — Employment Contract")
print("=" * 70)

if report:
    level = report.get("overall_risk_rating", "?")
    badge = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(level, "⚪")

    print(f"\nOverall Risk: {badge} {level.upper()}")
    print(f"\nExecutive Summary:")
    wrap_print(f"  {report.get('executive_summary', '')}")

    print(f"\nTop Risks:")
    for r in report.get("top_risks", []):
        print(f"  🔴 {r}")

    print(f"\nRecommended Actions:")
    for a in report.get("recommended_actions", []):
        print(f"  → {a}")

    print(f"\n⚠️ {report.get('disclaimer', 'Not legal advice.')}")
else:
    wrap_print(resp[:500])

### Mini Challenge

1. **Clause comparison tool:** Build a function that takes two contracts and highlights differences in equivalent clause types (e.g., compare both termination clauses side-by-side). Use the LLM to explain which version is more favorable to each party.

2. **Obligation extraction:** For each contract, extract all concrete obligations (deadlines, dollar amounts, percentages, notice periods) into a structured timeline. This is a pure information extraction task — separate from risk assessment.

3. **Custom risk profile:** Define a "buyer-friendly" and "seller-friendly" scoring rubric. Score each clause on both scales. A clause that's 5/5 buyer-friendly might be 1/5 seller-friendly. This teaches perspective-dependent analysis.

4. **Clause library:** Build a library of "gold standard" clause templates for each category. When reviewing a new contract, retrieve the most similar template and highlight deviations. This is a practical tool for contract standardization.

## 29. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nContracts analyzed: {len(SAMPLE_CONTRACTS)}")
print(f"Clauses parsed: {len(all_clauses)}")
print(f"Chunks indexed: {len(all_chunks)}")

print(f"\nPipeline stages:")
print(f"  ✅ Stage 1: Contract parsing — headings → structured clauses")
print(f"  ✅ Stage 2: Embedding + ChromaDB indexing with metadata")
print(f"  ✅ Stage 3: Semantic retrieval with category/contract filters")
print(f"  ✅ Stage 4: LLM-based risk assessment (5 risk patterns)")
print(f"  ✅ Stage 5: Missing clause detection against checklists")
print(f"  ✅ Stage 6: RAG Q&A over contract library")

print(f"\nEvaluations:")
print(f"  ✅ Retrieval accuracy (contract + category)")
print(f"  ✅ Risk assessment consistency (3 trials)")
print(f"  ✅ Out-of-scope question handling")
print(f"  ✅ Gap analysis with LLM explanations")
print(f"  ✅ Cross-contract comparison")
print(f"  ✅ Red-flag executive report")

print(f"\n⚠️  Limitations acknowledged in Section 27:")
print(f"  • Not legal advice — always consult a qualified attorney")
print(f"  • Jurisdiction-blind — cannot assess enforceability")
print(f"  • Hallucination risk — always verify LLM citations")
print(f"  • Template bias — unusual but valid clauses may be flagged")

## 30. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Contract parsing** splits on clause headings and enriches each clause with metadata (contract, category, parties) |
| 2 | **Clause categories** (liability, IP, termination, etc.) enable filtered retrieval — "show me only IP clauses" |
| 3 | **Risk assessment** uses pattern matching: vague language, one-sided terms, broad scope, missing protections |
| 4 | **Missing clause detection** compares against standard checklists — gaps are as risky as bad clauses |
| 5 | **RAG Q&A** lets you ask questions across multiple contracts with cited answers |
| 6 | **Consistency matters** — if the LLM gives different risk ratings on repeated runs, don't trust it alone |
| 7 | **Limitations are features** — knowing what the system CAN'T do prevents dangerous misuse |
| 8 | **Always recommend a lawyer** — this tool is a pre-screening aid, not a replacement for legal counsel |
| 9 | **Low temperature (0.1)** keeps analysis faithful — high temperature produces creative but unreliable legal text |

---

*This notebook is for **educational purposes only** and is not legal advice. Always consult a qualified attorney for real contract review.*

*Part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*